In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [2]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [3]:
API_KEY    = census_key  
STATE_FIPS = "54"                         
YEARS      = [2019, 2020, 2021, 2022, 2023]

VARIABLE   = "S2301_C02_001E,S2301_C04_001E"

records = []

In [4]:
for year in YEARS:
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5/subject"
        f"?get=NAME,{VARIABLE}"
        f"&for=county:*"
        f"&in=state:{STATE_FIPS}"
        f"&key={API_KEY}"
    )
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    headers = data[0]
    for row in data[1:]:
        record = dict(zip(headers, row))
        record["Year"] = year
        records.append(record)

df = pd.DataFrame(records)

df


,NAME,S2301_C02_001E,S2301_C04_001E,state,county,Year
0,"Summers County, West Virginia",44.1,11.7,54,089,2019
1,"Greenbrier County, West Virginia",50.7,5.7,54,025,2019
2,"Mineral County, West Virginia",54.6,5.6,54,057,2019
3,"Lewis County, West Virginia",50.6,6.8,54,041,2019
4,"Pocahontas County, West Virginia",49.1,4.4,54,075,2019
...,...,...,...,...,...,...
270,"Webster County, West Virginia",41.0,9.6,54,101,2023
271,"Wetzel County, West Virginia",46.6,5.8,54,103,2023
272,"Wirt County, West Virginia",48.1,3.3,54,105,2023
273,"Wood County, West Virginia",55.5,5.6,54,107,2023


In [5]:
# ── STEP 2: Clean up columns ─────────────────────────────────────────────────
df["FIPS_Code"]                      = df["state"] + df["county"]
df["County"]                         = df["NAME"].str.replace(", West Virginia", "", regex=False)
df["Labor_Force_Participation_Rate"] = pd.to_numeric(df["S2301_C02_001E"], errors="coerce")
df["Unemployment_Rate"]              = pd.to_numeric(df["S2301_C04_001E"], errors="coerce")

# ── STEP 3: Final structure ───────────────────────────────────────────────────
df = df[["Year", "FIPS_Code", "County", "Labor_Force_Participation_Rate", "Unemployment_Rate"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

df

,Year,FIPS_Code,County,Labor_Force_Participation_Rate,Unemployment_Rate
0,2019,54001,Barbour County,52.6,7.6
1,2019,54003,Berkeley County,65.4,6.0
2,2019,54005,Boone County,40.3,10.6
3,2019,54007,Braxton County,52.0,14.4
4,2019,54009,Brooke County,55.5,3.8
...,...,...,...,...,...
270,2023,54101,Webster County,41.0,9.6
271,2023,54103,Wetzel County,46.6,5.8
272,2023,54105,Wirt County,48.1,3.3
273,2023,54107,Wood County,55.5,5.6


In [6]:
df_rucc  = pd.read_csv("../data/RUCC/Ruralurbancontinuumcodes2023.csv", encoding="latin-1", dtype={"FIPS": str})
df_rucc


,FIPS,State,County_Name,Attribute,Value
0,01001,AL,Autauga County,Population_2020,58805
1,01001,AL,Autauga County,RUCC_2023,2
2,01001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."
3,01003,AL,Baldwin County,Population_2020,231767
4,01003,AL,Baldwin County,RUCC_2023,3
...,...,...,...,...,...
9698,78020,VI,St. John Island,RUCC_2023,9
9699,78020,VI,St. John Island,Description,"Nonmetro - Urban population of fewer than 5,00..."
9700,78030,VI,St. Thomas Island,Population_2020,42261
9701,78030,VI,St. Thomas Island,RUCC_2023,5


In [7]:
df_rucc = df_rucc.pivot_table(index=["FIPS", "State", "County_Name"], 
                               columns="Attribute", 
                               values="Value", 
                               aggfunc="first").reset_index()

In [8]:
df_rucc.columns.name = None

# Keep only what you need
df_rucc = df_rucc[["FIPS", "County_Name", "RUCC_2023"]].copy()

# Standardize FIPS to 5 digits and filter to WV
df_rucc["FIPS_Code"] = df_rucc["FIPS"].str.zfill(5)
df_rucc_wv = df_rucc[df_rucc["FIPS_Code"].str.startswith("54")][
    ["FIPS_Code", "RUCC_2023"]
].copy()
df_rucc_wv.rename(columns={"RUCC_2023": "Rural_Urban_Continuum_Code_2023"}, inplace=True)

In [9]:
# ── STEP 3: Merge RUCC onto existing dataframe ───────────────────────────────
df = pd.merge(df, df_rucc_wv, on="FIPS_Code", how="left")

# ── STEP 4: Final structure ───────────────────────────────────────────────────
df = df[["Year", "FIPS_Code", "County", "Labor_Force_Participation_Rate",
         "Unemployment_Rate", "Rural_Urban_Continuum_Code_2023"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

In [10]:
df

,Year,FIPS_Code,County,Labor_Force_Participation_Rate,Unemployment_Rate,Rural_Urban_Continuum_Code_2023
0,2019,54001,Barbour County,52.6,7.6,9
1,2019,54003,Berkeley County,65.4,6.0,2
2,2019,54005,Boone County,40.3,10.6,3
3,2019,54007,Braxton County,52.0,14.4,8
4,2019,54009,Brooke County,55.5,3.8,3
...,...,...,...,...,...,...
270,2023,54101,Webster County,41.0,9.6,9
271,2023,54103,Wetzel County,46.6,5.8,7
272,2023,54105,Wirt County,48.1,3.3,3
273,2023,54107,Wood County,55.5,5.6,3


In [11]:
df.to_csv("unemployment_data.csv", index=False)